# 05 · Cypher at scale — read-only on a graph this demo did not build

Points the cypher server (read-only) at the pre-existing `arxiv` graph (Papers/Authors/Categories/Years). Schema introspection, big aggregations, plan inspection, index advice, health, the top statements, oversized-property sanitization and pagination all hold up.

It creates nothing: not the database, not the graph. Skips cleanly if the graph isn't present.

In [1]:
import warnings; warnings.filterwarnings("ignore")   # quiet 3rd-party import warnings
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # examples/demos
from _common import clients, console
print("helpers ready — no API key needed (the MCP servers are pure tools)")

helpers ready — no API key needed (the MCP servers are pure tools)


## Schema, and the same aggregation two ways

Both queries answer "papers by year". One reads `p.year` off every Paper; the other reads the `Year` node this graph already models. The answers are identical and the times are not.

In [2]:
import time
from _common import config

PROPERTY_FORM = ('MATCH (p:"Paper") WHERE p.year IS NOT NULL '
                 'RETURN p.year AS year, count(*) AS papers ORDER BY year DESC LIMIT 5')
EDGE_FORM = ('MATCH (:"Paper")-[:"UPDATED_IN"]->(y:"Year") '
             'RETURN y.year AS year, count(*) AS papers ORDER BY year DESC LIMIT 5')

async def timed(cy, query):
    start = time.perf_counter()
    rows = clients.data(await cy.call_tool("read_agensgraph_cypher", {"query": query}))
    return rows["rows"], (time.perf_counter() - start) * 1000

async def scale():
    if not config.graph_exists("agensgraph_demos", "arxiv"):
        print("arxiv graph not present — skipping (the flights demos still run)."); return
    # read_only=True: the write tool is withheld, and the client creates neither the
    # database nor the graph — this is somebody else's data.
    async with clients.cypher_client("agensgraph_demos", "arxiv", read_only=True) as cy:
        console.kv("tools", sorted(t.name for t in await cy.list_tools()))
        schema = clients.data(await cy.call_tool("get_agensgraph_schema", {}))
        for label, info in schema.items():
            console.kv(label, f"{info.get('count'):,} nodes")

        by_property, property_ms = await timed(cy, PROPERTY_FORM)
        by_edge, edge_ms = await timed(cy, EDGE_FORM)
        console.table([(r["year"], r["papers"]) for r in by_edge], headers=["year", "papers"])
        console.kv("reading p.year", f"{property_ms:,.0f} ms")
        console.kv("reading the Year node", f"{edge_ms:,.1f} ms")
        console.kv("same answer", by_property == by_edge)

        p = clients.data(await cy.call_tool("read_agensgraph_cypher",
                                            {"query": 'MATCH (p:"Paper") RETURN p LIMIT 1'}))
        console.kv("Paper properties", list(p["rows"][0]["p"]["properties"]))
        console.kv("embedding", p["rows"][0]["p"]["properties"]["embedding"])
await scale()

  tools                      ['agensgraph_health', 'explain_agensgraph_cypher', 'get_agensgraph_schema', 'read_agensgraph_cypher', 'recommend_property_indexes', 'top_cypher_queries']


  Paper                      50,000 nodes
  Author                     88,455 nodes
  Category                   147 nodes
  Year                       17 nodes


  year  papers
  ----  ------
  2023  29    
  2022  62    
  2021  73    
  2020  107   
  2019  532   
  reading p.year             8,340 ms
  reading the Year node      9.3 ms
  same answer                True
  Paper properties           ['id', 'year', 'title', 'abstract', 'embedding']
  embedding                  <omitted: list of 1536 items>


## Plans, index advice, health, top statements — and who holds the read-only boundary

A write sent to the read tool is refused before it leaves this process, which is a good error message rather than a boundary. The same write handed to `explain_agensgraph_cypher` with `analyze` really is executed — and the read-only transaction refuses it with `25006`, leaving nothing behind.

In [3]:
WRITE = 'CREATE (:"Paper" {id: -1})'

async def tools_and_boundary():
    if not config.graph_exists("agensgraph_demos", "arxiv"):
        print("arxiv graph not present — skipping."); return
    async with clients.cypher_client("agensgraph_demos", "arxiv", read_only=True) as cy:
        plan = clients.data(await cy.call_tool("explain_agensgraph_cypher", {"query": PROPERTY_FORM}))
        node = plan[0]["Plan"]
        while node.get("Plans"):
            node = node["Plans"][0]
        console.kv("plan reaches", f"{node['Node Type']} on {node.get('Relation Name')}")

        advice = clients.data(await cy.call_tool("recommend_property_indexes",
            {"query": 'MATCH (p:"Paper") WHERE p.year = 2019 RETURN p.title AS title'}))
        for finding in advice["findings"]:
            console.kv(finding["kind"], finding["suggestion"])
        console.kv("costed against a built index", advice["verified"])

        health = clients.data(await cy.call_tool("agensgraph_health", {}))
        console.kv("health checks", sorted(health))
        console.kv("optional extensions", health["extensions"])

        top = clients.data(await cy.call_tool("top_cypher_queries", {"limit": 3}))
        console.kv("statements tracked", len(top) if isinstance(top, list) else top)

        with console.expecting_refusal():
            try:
                await cy.call_tool("read_agensgraph_cypher", {"query": WRITE})
            except Exception as e:
                console.kv("read tool", f"refused here: {str(e).split('.')[0]}")
            try:
                await cy.call_tool("explain_agensgraph_cypher", {"query": WRITE, "analyze": True})
            except Exception as e:
                console.kv("explain analyze", f"refused by the database: {str(e).split(' (')[0]}")
        left = clients.data(await cy.call_tool("read_agensgraph_cypher",
            {"query": 'MATCH (p:"Paper") WHERE p.id = -1 RETURN count(*) AS n'}))
        console.kv("Papers left behind", left["rows"][0]["n"])
await tools_and_boundary()

  plan reaches               Seq Scan on Paper
  missing_index              CREATE PROPERTY INDEX ON "Paper" (year);
  costed against a built index False
  health checks              ['cache_hit_ratio', 'connections', 'extensions', 'graphmeta', 'pg_buffercache', 'unused_indexes', 'vacuum_age']
  optional extensions        {'pg_buffercache': False, 'pg_stat_statements': True, 'pgstattuple': True}
  statements tracked         3
  read tool                  refused here: This tool only reads
  explain analyze            refused by the database: Could not plan that statement: [25006] cannot write to a graph in a read-only transaction
  Papers left behind         0
